# Structured Output

Models can be requested to provide their repsonse in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supoorts muliple schema types and methods for enforcing structured output

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [70]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

groq = init_chat_model('groq:qwen/qwen3-32b')
chatgpt = init_chat_model('gpt-5.1')
chatgpt

ChatOpenAI(profile={'name': 'GPT-5.1', 'release_date': '2025-11-13', 'last_updated': '2025-11-13', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000238EAF90110>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000238EAF90650>, root_client=<openai.OpenAI object at 0x00000238EAF948C0>, root_async_client=<openai.AsyncOpenAI object at 0x00000238EAF949B0>, model_name='gpt-5.1', model_kwargs={}, openai_api_key=SecretStr('**

In [71]:
from pydantic import BaseModel, Field
from typing import List

class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")
    gender: str = Field(description="Gender of the person")

class Actor(BaseModel):
    name: str = Field(description="Name of the actor")
    age: int = Field(description="Age of the actor")
    gender: str = Field(description="Gender of the actor")

class Movie(BaseModel):
    title: str = Field(description = 'Title of movie')
    year: int = Field(description = 'year when movie was released')
    rating:int = Field(description= 'Rating of movie out of 5')
    actor: List[Person]  = Field(description='The Main actor and actress of the movie')
    director:Person = Field(description='Director of movie')
    


In [72]:
# groq_structured = groq.with_structured_output(Movie)
chatgpt_structured = chatgpt.with_structured_output(Movie) #add include_raw=True to get the raw response
chatgpt_structured.invoke('Provide details of movie mardani 2 ',)
# print(response)

Movie(title='Mardaani 2', year=2019, rating=4, actor=[Person(name='Rani Mukerji', age=46, gender='Female'), Person(name='Vishal Jethwa', age=30, gender='Male')], director=Person(name='Gopi Puthran', age=50, gender='Male'))

## Nested Structure

In [73]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name : str
    role: str
    gender: str
    age: str
class MovieDetails(BaseModel):
    title: str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description= 'Budget in indian rupee Crore')

In [74]:
chatgpt_structured = chatgpt.with_structured_output(MovieDetails)
chatgpt_structured.invoke('Provide details of movie Anora')

MovieDetails(title='Anora', year=2024, cast=[Actor(name='Mikey Madison', role='Anora', gender='Female', age='25-35'), Actor(name='Yura Borisov', role='Vanya (Ivan), the oligarch’s son', gender='Male', age='25-35'), Actor(name='Mark Eydelshteyn', role='Vanya’s father / Russian oligarch', gender='Male', age='50-65'), Actor(name='Edward Colon', role='Friend / supporting role', gender='Male', age='25-40'), Actor(name='Wayne Diamond', role='Nightlife figure / supporting role', gender='Male', age='60-75')], genres=['Drama', 'Comedy', 'Romance'], budget=None)

## TypedDict

TypedDict provides a simpler alternative using Python's built-in-typing, ideal when you don't need run time validation

In [75]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    #create details of movie
    title: Annotated[str,...,'The title of movie']
    cast: Annotated[str,...,'The cast of movie']
    mainLead:Annotated[str,...,'Main character']
    year: Annotated[int,...,'Year movie got released']
    director: Annotated[str,...,'Name of director']
    rating: Annotated[float,...,'IMDB Rating movie got out of 10']
    genre:Annotated[str,...,'genre of movie']
    budget:Annotated[int,...,'Budget of movie in inr crores']
    language:Annotated[str,...,'Language of movie']

In [76]:
chatgpt_typedDict = chatgpt.with_structured_output(MovieDict)
chatgpt_typedDict.invoke('Provide the details of movie anora')

{'title': 'Anora',
 'cast': 'Mikey Madison, Yura Borisov, Mark Eidelstein, Edoardo Ballerini, Karren Karagulian',
 'mainLead': 'Anora (played by Mikey Madison)',
 'year': 2024,
 'director': 'Sean Baker',
 'rating': 7.7,
 'genre': 'Drama, Comedy, Romance',
 'budget': 83,
 'language': 'English'}

In [77]:
groq.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

# Data Classes

A data class is a class typically containing mainly data, although tere aren't really any restrictions. You create it using the @dataclass decorator

In [78]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description = 'The name of the person add mr or mrs based on their gender and if there is no gender consider male')
    email: str = Field(description = 'The email id of the person')
    phone: str = Field(description = 'The phone number of the person')


chatGPT = create_agent(model='gpt-5.1',response_format=ContactInfo) #Auto selects provider strategy

response = chatGPT.invoke({'messages':[{'role':'user','content':'Extract information from Mohammad Imran Khan, ikik790@gmail.com, 8039290382'}]})
response['structured_response']

ContactInfo(name='Mr Mohammad Imran Khan', email='ikik790@gmail.com', phone='8039290382')

In [81]:
## DataClass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description = 'The name of the person add mr or mrs based on their gender and if there is no gender consider male')
    email: str = Field(description = 'The email id of the person')
    phone: str = Field(description = 'The phone number of the person')

agent = create_agent(model='gpt-5.1',response_format=ContactInfo)

agent.invoke({'messages':[{'role':'user','content':'Extract information from Mohammad Imran Khan, ikik790@gmail.com, 8039290382'}]})['structured_response']

ContactInfo(name='Mr Mohammad Imran Khan', email='ikik790@gmail.com', phone='8039290382')